# Premier League Match Prediction - Feature Engineering

## Objective

This notebook creates pre-match features for predicting Premier League match outcomes. The EDA showed that home advantage, team strength, goals, shot volume, shot quality, and season-level context are relevant signals. The key requirement is that every feature must be known before kickoff.

The final output is a match-level modeling dataset where each row represents one fixture and the target is `FTR`.

## 1. Data Leakage Rule

The model should not use same-match values such as final goals, half-time goals, shots, shots on target, corners, fouls, or cards from the match being predicted. These values are only known after kickoff.

This notebook uses those columns only to calculate historical rolling statistics from previous matches.

## 2. Load Cleaned Match Data

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

DATA_PATH = Path("../data/processed/epl_combined_cleaned.csv")
OUTPUT_PATH = Path("../data/processed/epl_features.csv")

df = pd.read_csv(DATA_PATH)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 3800
Columns: 25


,Season,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
0,2016_2017,E0,2016-08-13,NaN,Burnley,Swansea,0,1,A,0,0,D,J Moss,10,17,3,9,10,14,7,4,3,2,0,0
1,2016_2017,E0,2016-08-13,NaN,Crystal Palace,West Brom,0,1,A,0,0,D,C Pawson,14,13,4,3,12,15,3,6,2,2,0,0
2,2016_2017,E0,2016-08-13,NaN,Everton,Tottenham,1,1,D,1,0,H,M Atkinson,12,13,6,4,10,14,5,6,0,0,0,0
3,2016_2017,E0,2016-08-13,NaN,Hull,Leicester,2,1,H,1,0,H,M Dean,14,18,5,5,8,17,5,3,2,2,0,0
4,2016_2017,E0,2016-08-13,NaN,Man City,Sunderland,2,1,H,1,0,H,R Madley,16,7,4,3,11,14,9,6,1,2,0,0


## 3. Prepare Chronological Match Order

Rolling features must be calculated in chronological order. Some older seasons are missing kickoff time, so missing times are filled with `00:00` for sorting. This does not create a predictive feature; it only gives a stable ordering within each match date.

In [2]:
matches = df.copy()

matches["Date"] = pd.to_datetime(matches["Date"], errors="coerce")
matches["Time_For_Sort"] = matches["Time"].fillna("00:00")
matches["Kickoff"] = pd.to_datetime(
    matches["Date"].dt.strftime("%Y-%m-%d") + " " + matches["Time_For_Sort"],
    errors="coerce"
)

matches = matches.sort_values(["Kickoff", "HomeTeam", "AwayTeam"]).reset_index(drop=True)
matches["MatchID"] = np.arange(len(matches))

matches[["MatchID", "Season", "Date", "Time", "HomeTeam", "AwayTeam", "FTR"]].head()

,MatchID,Season,Date,Time,HomeTeam,AwayTeam,FTR
0,0,2016_2017,2016-08-13,NaN,Burnley,Swansea,A
1,1,2016_2017,2016-08-13,NaN,Crystal Palace,West Brom,A
2,2,2016_2017,2016-08-13,NaN,Everton,Tottenham,D
3,3,2016_2017,2016-08-13,NaN,Hull,Leicester,H
4,4,2016_2017,2016-08-13,NaN,Man City,Sunderland,H


## 4. Create Target Variables

`FTR` remains the main multiclass target. Additional binary targets can be useful for baseline models or betting-style analysis.

In [3]:
target_map = {"H": 0, "D": 1, "A": 2}

matches["target_result"] = matches["FTR"]
matches["target_result_encoded"] = matches["FTR"].map(target_map)
matches["target_home_win"] = (matches["FTR"] == "H").astype(int)
matches["target_draw"] = (matches["FTR"] == "D").astype(int)
matches["target_away_win"] = (matches["FTR"] == "A").astype(int)

matches[["FTR", "target_result_encoded", "target_home_win", "target_draw", "target_away_win"]].head()

,FTR,target_result_encoded,target_home_win,target_draw,target_away_win
0,A,2,0,0,1
1,A,2,0,0,1
2,D,1,0,1,0
3,H,0,1,0,0
4,H,0,1,0,0


## 5. Build Team-Match Table

The source data has one row per fixture. Rolling team features are easier to calculate after converting each fixture into two team-level rows: one for the home team and one for the away team.

In [4]:
home_rows = matches[[
    "MatchID", "Season", "Date", "Kickoff", "HomeTeam", "AwayTeam",
    "FTHG", "FTAG", "FTR", "HS", "AS", "HST", "AST", "HF", "AF",
    "HC", "AC", "HY", "AY", "HR", "AR"
]].copy()

home_rows = home_rows.rename(columns={
    "HomeTeam": "Team",
    "AwayTeam": "Opponent",
    "FTHG": "goals_for",
    "FTAG": "goals_against",
    "HS": "shots_for",
    "AS": "shots_against",
    "HST": "sot_for",
    "AST": "sot_against",
    "HF": "fouls_for",
    "AF": "fouls_against",
    "HC": "corners_for",
    "AC": "corners_against",
    "HY": "yellow_for",
    "AY": "yellow_against",
    "HR": "red_for",
    "AR": "red_against",
})
home_rows["Venue"] = "Home"
home_rows["result"] = home_rows["FTR"].map({"H": "W", "D": "D", "A": "L"})

away_rows = matches[[
    "MatchID", "Season", "Date", "Kickoff", "AwayTeam", "HomeTeam",
    "FTAG", "FTHG", "FTR", "AS", "HS", "AST", "HST", "AF", "HF",
    "AC", "HC", "AY", "HY", "AR", "HR"
]].copy()

away_rows = away_rows.rename(columns={
    "AwayTeam": "Team",
    "HomeTeam": "Opponent",
    "FTAG": "goals_for",
    "FTHG": "goals_against",
    "AS": "shots_for",
    "HS": "shots_against",
    "AST": "sot_for",
    "HST": "sot_against",
    "AF": "fouls_for",
    "HF": "fouls_against",
    "AC": "corners_for",
    "HC": "corners_against",
    "AY": "yellow_for",
    "HY": "yellow_against",
    "AR": "red_for",
    "HR": "red_against",
})
away_rows["Venue"] = "Away"
away_rows["result"] = away_rows["FTR"].map({"H": "L", "D": "D", "A": "W"})

team_matches = pd.concat([home_rows, away_rows], ignore_index=True)
team_matches = team_matches.sort_values(["Team", "Kickoff", "MatchID"]).reset_index(drop=True)

team_matches["points"] = team_matches["result"].map({"W": 3, "D": 1, "L": 0})
team_matches["win"] = (team_matches["result"] == "W").astype(int)
team_matches["draw"] = (team_matches["result"] == "D").astype(int)
team_matches["loss"] = (team_matches["result"] == "L").astype(int)
team_matches["goal_diff"] = team_matches["goals_for"] - team_matches["goals_against"]
team_matches["shot_diff"] = team_matches["shots_for"] - team_matches["shots_against"]
team_matches["sot_diff"] = team_matches["sot_for"] - team_matches["sot_against"]
team_matches["corner_diff"] = team_matches["corners_for"] - team_matches["corners_against"]

team_matches.head()

,MatchID,Season,Date,Kickoff,Team,Opponent,goals_for,goals_against,FTR,shots_for,shots_against,sot_for,sot_against,fouls_for,fouls_against,corners_for,corners_against,yellow_for,yellow_against,red_for,red_against,Venue,result,points,win,draw,loss,goal_diff,shot_diff,sot_diff,corner_diff
0,7,2016_2017,2016-08-14,2016-08-14,Arsenal,Liverpool,3,4,A,9,16,5,7,13,17,5,4,3,3,0,0,Home,L,0,0,0,1,-1,-7,-2,1
1,12,2016_2017,2016-08-20,2016-08-20,Arsenal,Leicester,0,0,D,13,8,4,1,7,11,7,2,2,1,0,0,Away,D,1,0,1,0,0,5,3,5
2,27,2016_2017,2016-08-27,2016-08-27,Arsenal,Watford,3,1,A,10,14,7,6,15,18,2,3,1,6,0,0,Away,W,3,1,0,0,2,-4,1,-1
3,30,2016_2017,2016-09-10,2016-09-10,Arsenal,Southampton,2,1,H,17,11,2,5,10,8,6,1,2,5,0,0,Home,W,3,1,0,0,1,6,-3,5
4,42,2016_2017,2016-09-17,2016-09-17,Arsenal,Hull,4,1,A,24,6,9,2,12,9,4,2,2,0,0,1,Away,W,3,1,0,0,3,18,7,2


## 6. Rolling Team Form Features

These features summarize each team's previous matches. The `shift(1)` step is critical: it excludes the current match before calculating rolling values.

In [5]:
rolling_base_cols = [
    "points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff",
    "shots_for", "shots_against", "shot_diff", "sot_for", "sot_against", "sot_diff",
    "corners_for", "corners_against", "corner_diff", "fouls_for", "fouls_against",
    "yellow_for", "yellow_against", "red_for", "red_against"
]

def add_rolling_features(data, group_cols, value_cols, windows, prefix):
    data = data.sort_values(group_cols + ["Kickoff", "MatchID"]).copy()
    grouped = data.groupby(group_cols, group_keys=False)

    for window in windows:
        for col in value_cols:
            feature_name = f"{prefix}_{col}_last_{window}"
            data[feature_name] = grouped[col].transform(
                lambda s: s.shift(1).rolling(window=window, min_periods=1).mean()
            )

    return data

team_features = add_rolling_features(
    data=team_matches,
    group_cols=["Team"],
    value_cols=rolling_base_cols,
    windows=[5, 10],
    prefix="overall"
)

team_features[[
    "Team", "Date", "Venue", "Opponent", "points",
    "overall_points_last_5", "overall_goal_diff_last_5",
    "overall_sot_diff_last_5"
]].head(12)

,Team,Date,Venue,Opponent,points,overall_points_last_5,overall_goal_diff_last_5,overall_sot_diff_last_5
0,Arsenal,2016-08-14,Home,Liverpool,0,NaN,NaN,NaN
1,Arsenal,2016-08-20,Away,Leicester,1,0.000000,-1.000000,-2.000000
2,Arsenal,2016-08-27,Away,Watford,3,0.500000,-0.500000,0.500000
3,Arsenal,2016-09-10,Home,Southampton,3,1.333333,0.333333,0.666667
4,Arsenal,2016-09-17,Away,Hull,3,1.750000,0.500000,-0.250000
5,Arsenal,2016-09-24,Home,Chelsea,3,2.000000,1.000000,1.200000
6,Arsenal,2016-10-02,Away,Burnley,3,2.600000,1.800000,2.200000
7,Arsenal,2016-10-15,Home,Swansea,3,3.000000,2.000000,2.000000
8,Arsenal,2016-10-22,Home,Middlesbrough,1,3.000000,1.800000,1.800000
9,Arsenal,2016-10-29,Away,Sunderland,3,2.600000,1.600000,2.600000


## 7. Home and Away Form Features

The EDA showed a clear home advantage, so venue-specific form should be modeled separately. Home rows receive prior home-match features, and away rows receive prior away-match features.

In [6]:
venue_value_cols = [
    "points", "win", "goals_for", "goals_against", "goal_diff",
    "shots_for", "shots_against", "sot_for", "sot_against"
]

team_features = add_rolling_features(
    data=team_features,
    group_cols=["Team", "Venue"],
    value_cols=venue_value_cols,
    windows=[5],
    prefix="venue"
)

team_features[[
    "Team", "Date", "Venue", "Opponent",
    "venue_points_last_5", "venue_goal_diff_last_5",
    "venue_sot_for_last_5", "venue_sot_against_last_5"
]].head(12)

,Team,Date,Venue,Opponent,venue_points_last_5,venue_goal_diff_last_5,venue_sot_for_last_5,venue_sot_against_last_5
1,Arsenal,2016-08-20,Away,Leicester,NaN,NaN,NaN,NaN
2,Arsenal,2016-08-27,Away,Watford,1.000000,0.000000,4.000000,1.0
4,Arsenal,2016-09-17,Away,Hull,2.000000,1.000000,5.500000,3.5
6,Arsenal,2016-10-02,Away,Burnley,2.333333,1.666667,6.666667,3.0
9,Arsenal,2016-10-29,Away,Sunderland,2.500000,1.500000,5.750000,2.5
11,Arsenal,2016-11-19,Away,Man United,2.600000,1.800000,6.000000,2.2
13,Arsenal,2016-12-03,Away,West Ham,2.600000,1.800000,5.400000,3.0
15,Arsenal,2016-12-13,Away,Everton,2.600000,2.200000,6.000000,2.4
16,Arsenal,2016-12-18,Away,Man City,2.000000,1.400000,4.800000,2.8
19,Arsenal,2017-01-03,Away,Bournemouth,1.400000,1.000000,4.400000,3.6


## 8. Rest and Schedule Features

Rest days and fixture congestion can affect match performance. These features use only each team's previous match dates.

In [7]:
team_features = team_features.sort_values(["Team", "Kickoff", "MatchID"]).copy()

team_features["team_matches_played_before"] = team_features.groupby("Team").cumcount()
team_features["days_since_last_match"] = (
    team_features.groupby("Team")["Kickoff"].diff().dt.days
)

team_features[["Team", "Date", "Opponent", "team_matches_played_before", "days_since_last_match"]].head(12)

,Team,Date,Opponent,team_matches_played_before,days_since_last_match
0,Arsenal,2016-08-14,Liverpool,0,NaN
1,Arsenal,2016-08-20,Leicester,1,6.0
2,Arsenal,2016-08-27,Watford,2,7.0
3,Arsenal,2016-09-10,Southampton,3,14.0
4,Arsenal,2016-09-17,Hull,4,7.0
5,Arsenal,2016-09-24,Chelsea,5,7.0
6,Arsenal,2016-10-02,Burnley,6,8.0
7,Arsenal,2016-10-15,Swansea,7,13.0
8,Arsenal,2016-10-22,Middlesbrough,8,7.0
9,Arsenal,2016-10-29,Sunderland,9,7.0


The `days_since_last_match` feature is included directly. Fixture congestion features such as matches played in the last 14 days can be added later, but they are left out of the first feature file to keep the initial pipeline simple and auditable.

## 9. Merge Team Features Back to Match Level

The model needs one row per fixture. This section merges the historical features for the home team and away team onto the original match row.

In [8]:
feature_cols = [
    col for col in team_features.columns
    if col.startswith("overall_")
    or col.startswith("venue_")
    or col in ["MatchID", "Team", "Venue", "team_matches_played_before", "days_since_last_match"]
]

home_features = team_features[team_features["Venue"] == "Home"][feature_cols].copy()
away_features = team_features[team_features["Venue"] == "Away"][feature_cols].copy()

home_features = home_features.drop(columns=["Venue"]).add_prefix("home_")
home_features = home_features.rename(columns={"home_MatchID": "MatchID", "home_Team": "HomeTeam"})

away_features = away_features.drop(columns=["Venue"]).add_prefix("away_")
away_features = away_features.rename(columns={"away_MatchID": "MatchID", "away_Team": "AwayTeam"})

model_df = matches.merge(home_features, on=["MatchID", "HomeTeam"], how="left")
model_df = model_df.merge(away_features, on=["MatchID", "AwayTeam"], how="left")

print("Rows:", model_df.shape[0])
print("Columns:", model_df.shape[1])

model_df.head()

Rows: 3800
Columns: 143


,Season,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR,Time_For_Sort,Kickoff,MatchID,target_result,target_result_encoded,target_home_win,target_draw,target_away_win,home_overall_points_last_5,home_overall_win_last_5,home_overall_draw_last_5,home_overall_loss_last_5,home_overall_goals_for_last_5,home_overall_goals_against_last_5,home_overall_goal_diff_last_5,home_overall_shots_for_last_5,home_overall_shots_against_last_5,home_overall_shot_diff_last_5,home_overall_sot_for_last_5,home_overall_sot_against_last_5,home_overall_sot_diff_last_5,home_overall_corners_for_last_5,home_overall_corners_against_last_5,home_overall_corner_diff_last_5,home_overall_fouls_for_last_5,home_overall_fouls_against_last_5,home_overall_yellow_for_last_5,home_overall_yellow_against_last_5,home_overall_red_for_last_5,home_overall_red_against_last_5,home_overall_points_last_10,home_overall_win_last_10,home_overall_draw_last_10,home_overall_loss_last_10,home_overall_goals_for_last_10,home_overall_goals_against_last_10,home_overall_goal_diff_last_10,home_overall_shots_for_last_10,home_overall_shots_against_last_10,home_overall_shot_diff_last_10,home_overall_sot_for_last_10,home_overall_sot_against_last_10,home_overall_sot_diff_last_10,home_overall_corners_for_last_10,home_overall_corners_against_last_10,home_overall_corner_diff_last_10,home_overall_fouls_for_last_10,home_overall_fouls_against_last_10,home_overall_yellow_for_last_10,home_overall_yellow_against_last_10,home_overall_red_for_last_10,home_overall_red_against_last_10,home_venue_points_last_5,home_venue_win_last_5,home_venue_goals_for_last_5,home_venue_goals_against_last_5,home_venue_goal_diff_last_5,home_venue_shots_for_last_5,home_venue_shots_against_last_5,home_venue_sot_for_last_5,home_venue_sot_against_last_5,home_team_matches_played_before,home_days_since_last_match,away_overall_points_last_5,away_overall_win_last_5,away_overall_draw_last_5,away_overall_loss_last_5,away_overall_goals_for_last_5,away_overall_goals_against_last_5,away_overall_goal_diff_last_5,away_overall_shots_for_last_5,away_overall_shots_against_last_5,away_overall_shot_diff_last_5,away_overall_sot_for_last_5,away_overall_sot_against_last_5,away_overall_sot_diff_last_5,away_overall_corners_for_last_5,away_overall_corners_against_last_5,away_overall_corner_diff_last_5,away_overall_fouls_for_last_5,away_overall_fouls_against_last_5,away_overall_yellow_for_last_5,away_overall_yellow_against_last_5,away_overall_red_for_last_5,away_overall_red_against_last_5,away_overall_points_last_10,away_overall_win_last_10,away_overall_draw_last_10,away_overall_loss_last_10,away_overall_goals_for_last_10,away_overall_goals_against_last_10,away_overall_goal_diff_last_10,away_overall_shots_for_last_10,away_overall_shots_against_last_10,away_overall_shot_diff_last_10,away_overall_sot_for_last_10,away_overall_sot_against_last_10,away_overall_sot_diff_last_10,away_overall_corners_for_last_10,away_overall_corners_against_last_10,away_overall_corner_diff_last_10,away_overall_fouls_for_last_10,away_overall_fouls_against_last_10,away_overall_yellow_for_last_10,away_overall_yellow_against_last_10,away_overall_red_for_last_10,away_overall_red_against_last_10,away_venue_points_last_5,away_venue_win_last_5,away_venue_goals_for_last_5,away_venue_goals_against_last_5,away_venue_goal_diff_last_5,away_venue_shots_for_last_5,away_venue_shots_against_last_5,away_venue_sot_for_last_5,away_venue_sot_against_last_5,away_team_matches_played_before,away_days_since_last_match
0,2016_2017,E0,2016-08-13,NaN,Burnley,Swansea,0,1,A,0,0,D,J Moss,10,17,3,9,10,14,7,4,3,2,0,0,00:00,2016-08-13,0,A,2,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

## 10. Create Home-vs-Away Differential Features

Differential features compare the home team's historical strength to the away team's historical strength. These are often more useful than raw team features because they represent the matchup.

In [9]:
paired_metrics = [
    "overall_points_last_5",
    "overall_points_last_10",
    "overall_win_last_5",
    "overall_goal_diff_last_5",
    "overall_goal_diff_last_10",
    "overall_goals_for_last_5",
    "overall_goals_against_last_5",
    "overall_shot_diff_last_5",
    "overall_sot_diff_last_5",
    "overall_corner_diff_last_5",
    "venue_points_last_5",
    "venue_goal_diff_last_5",
    "venue_sot_for_last_5",
    "venue_sot_against_last_5",
    "team_matches_played_before",
    "days_since_last_match",
]

for metric in paired_metrics:
    home_col = f"home_{metric}"
    away_col = f"away_{metric}"

    if home_col in model_df.columns and away_col in model_df.columns:
        model_df[f"diff_{metric}"] = model_df[home_col] - model_df[away_col]

diff_cols = [col for col in model_df.columns if col.startswith("diff_")]

model_df[["HomeTeam", "AwayTeam", "FTR"] + diff_cols[:10]].head()

,HomeTeam,AwayTeam,FTR,diff_overall_points_last_5,diff_overall_points_last_10,diff_overall_win_last_5,diff_overall_goal_diff_last_5,diff_overall_goal_diff_last_10,diff_overall_goals_for_last_5,diff_overall_goals_against_last_5,diff_overall_shot_diff_last_5,diff_overall_sot_diff_last_5,diff_overall_corner_diff_last_5
0,Burnley,Swansea,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Crystal Palace,West Brom,A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Everton,Tottenham,D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Hull,Leicester,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Man City,Sunderland,H,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 11. Add Season Timing Features

The EDA showed that scoring environment and outcome rates can vary by season. These date-based features give the model basic season context.

In [10]:
model_df["match_month"] = model_df["Date"].dt.month
model_df["match_year"] = model_df["Date"].dt.year

model_df["season_match_number"] = model_df.groupby("Season").cumcount() + 1
model_df["season_progress"] = model_df["season_match_number"] / model_df.groupby("Season")["MatchID"].transform("count")

model_df["season_stage"] = pd.cut(
    model_df["season_progress"],
    bins=[0, 1/3, 2/3, 1],
    labels=["early", "middle", "late"],
    include_lowest=True
)

model_df[["Season", "Date", "match_month", "season_match_number", "season_progress", "season_stage"]].head()

,Season,Date,match_month,season_match_number,season_progress,season_stage
0,2016_2017,2016-08-13,8,1,0.002632,early
1,2016_2017,2016-08-13,8,2,0.005263,early
2,2016_2017,2016-08-13,8,3,0.007895,early
3,2016_2017,2016-08-13,8,4,0.010526,early
4,2016_2017,2016-08-13,8,5,0.013158,early


## 12. Add Referee Tendency Features

The processed dataset includes `Referee`. Referee assignment is known before kickoff, so historical referee tendencies can be valid pre-match features. These features summarize each referee's prior matches only.

In [11]:
referee_history = matches[["MatchID", "Kickoff", "Referee", "HF", "AF", "HY", "AY", "HR", "AR"]].copy()
referee_history["total_fouls"] = referee_history["HF"] + referee_history["AF"]
referee_history["total_yellow_cards"] = referee_history["HY"] + referee_history["AY"]
referee_history["total_red_cards"] = referee_history["HR"] + referee_history["AR"]
referee_history = referee_history.sort_values(["Referee", "Kickoff", "MatchID"])

referee_history["referee_matches_before"] = referee_history.groupby("Referee").cumcount()

for col in ["total_fouls", "total_yellow_cards", "total_red_cards"]:
    referee_history[f"referee_{col}_last_20"] = (
        referee_history.groupby("Referee")[col]
        .transform(lambda s: s.shift(1).rolling(window=20, min_periods=1).mean())
    )

referee_feature_cols = [
    "MatchID", "referee_matches_before", "referee_total_fouls_last_20",
    "referee_total_yellow_cards_last_20", "referee_total_red_cards_last_20"
]

model_df = model_df.merge(referee_history[referee_feature_cols], on="MatchID", how="left")

model_df[["Referee"] + referee_feature_cols[1:]].head(12)

,Referee,referee_matches_before,referee_total_fouls_last_20,referee_total_yellow_cards_last_20,referee_total_red_cards_last_20
0,J Moss,0,NaN,NaN,NaN
1,C Pawson,0,NaN,NaN,NaN
2,M Atkinson,0,NaN,NaN,NaN
3,M Dean,0,NaN,NaN,NaN
4,R Madley,0,NaN,NaN,NaN
5,K Friend,0,NaN,NaN,NaN
6,R East,0,NaN,NaN,NaN
7,M Oliver,0,NaN,NaN,NaN
8,A Marriner,0,NaN,NaN,NaN
9,A Taylor,0,NaN,NaN,NaN


## 13. Review Missing Feature Values

The first matches for each team will naturally have missing rolling features because there is no prior history in the dataset. These rows can be handled with imputation, dropped from modeling, or retained with missing-value-aware models.

In [12]:
model_feature_cols = [
    col for col in model_df.columns
    if col.startswith("home_overall_")
    or col.startswith("away_overall_")
    or col.startswith("home_venue_")
    or col.startswith("away_venue_")
    or col.startswith("diff_")
    or col.startswith("referee_")
    or col in [
        "home_team_matches_played_before", "away_team_matches_played_before",
        "home_days_since_last_match", "away_days_since_last_match",
        "match_month", "season_progress"
    ]
]

missing_summary = (
    model_df[model_feature_cols]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary[missing_summary > 0].head(20)

referee_total_red_cards_last_20       50
referee_total_yellow_cards_last_20    50
referee_total_fouls_last_20           50
diff_venue_sot_against_last_5         48
diff_venue_sot_for_last_5             48
diff_venue_goal_diff_last_5           48
diff_venue_points_last_5              48
home_venue_sot_against_last_5         34
away_venue_goal_diff_last_5           34
home_venue_shots_against_last_5       34
away_venue_goals_for_last_5           34
away_venue_win_last_5                 34
away_venue_points_last_5              34
home_venue_goals_for_last_5           34
home_venue_sot_for_last_5             34
home_venue_points_last_5              34
home_venue_win_last_5                 34
away_venue_shots_against_last_5       34
home_venue_goals_against_last_5       34
home_venue_goal_diff_last_5           34
dtype: int64

## 14. Build Final Modeling Dataset

The final dataset keeps match identifiers, target columns, pre-match features, and a few result columns for later evaluation. Same-match result statistics should not be used as model inputs.

In [13]:
id_cols = [
    "MatchID", "Season", "Date", "Time", "HomeTeam", "AwayTeam"
]

target_cols = [
    "target_result", "target_result_encoded", "target_home_win", "target_draw", "target_away_win"
]

evaluation_cols = ["FTHG", "FTAG", "FTR"]

final_cols = id_cols + target_cols + evaluation_cols + model_feature_cols
features_df = model_df[final_cols].copy()

print("Rows:", features_df.shape[0])
print("Columns:", features_df.shape[1])

features_df.head()

Rows: 3800
Columns: 146


,MatchID,Season,Date,Time,HomeTeam,AwayTeam,target_result,target_result_encoded,target_home_win,target_draw,target_away_win,FTHG,FTAG,FTR,home_overall_points_last_5,home_overall_win_last_5,home_overall_draw_last_5,home_overall_loss_last_5,home_overall_goals_for_last_5,home_overall_goals_against_last_5,home_overall_goal_diff_last_5,home_overall_shots_for_last_5,home_overall_shots_against_last_5,home_overall_shot_diff_last_5,home_overall_sot_for_last_5,home_overall_sot_against_last_5,home_overall_sot_diff_last_5,home_overall_corners_for_last_5,home_overall_corners_against_last_5,home_overall_corner_diff_last_5,home_overall_fouls_for_last_5,home_overall_fouls_against_last_5,home_overall_yellow_for_last_5,home_overall_yellow_against_last_5,home_overall_red_for_last_5,home_overall_red_against_last_5,home_overall_points_last_10,home_overall_win_last_10,home_overall_draw_last_10,home_overall_loss_last_10,home_overall_goals_for_last_10,home_overall_goals_against_last_10,home_overall_goal_diff_last_10,home_overall_shots_for_last_10,home_overall_shots_against_last_10,home_overall_shot_diff_last_10,home_overall_sot_for_last_10,home_overall_sot_against_last_10,home_overall_sot_diff_last_10,home_overall_corners_for_last_10,home_overall_corners_against_last_10,home_overall_corner_diff_last_10,home_overall_fouls_for_last_10,home_overall_fouls_against_last_10,home_overall_yellow_for_last_10,home_overall_yellow_against_last_10,home_overall_red_for_last_10,home_overall_red_against_last_10,home_venue_points_last_5,home_venue_win_last_5,home_venue_goals_for_last_5,home_venue_goals_against_last_5,home_venue_goal_diff_last_5,home_venue_shots_for_last_5,home_venue_shots_against_last_5,home_venue_sot_for_last_5,home_venue_sot_against_last_5,home_team_matches_played_before,home_days_since_last_match,away_overall_points_last_5,away_overall_win_last_5,away_overall_draw_last_5,away_overall_loss_last_5,away_overall_goals_for_last_5,away_overall_goals_against_last_5,away_overall_goal_diff_last_5,away_overall_shots_for_last_5,away_overall_shots_against_last_5,away_overall_shot_diff_last_5,away_overall_sot_for_last_5,away_overall_sot_against_last_5,away_overall_sot_diff_last_5,away_overall_corners_for_last_5,away_overall_corners_against_last_5,away_overall_corner_diff_last_5,away_overall_fouls_for_last_5,away_overall_fouls_against_last_5,away_overall_yellow_for_last_5,away_overall_yellow_against_last_5,away_overall_red_for_last_5,away_overall_red_against_last_5,away_overall_points_last_10,away_overall_win_last_10,away_overall_draw_last_10,away_overall_loss_last_10,away_overall_goals_for_last_10,away_overall_goals_against_last_10,away_overall_goal_diff_last_10,away_overall_shots_for_last_10,away_overall_shots_against_last_10,away_overall_shot_diff_last_10,away_overall_sot_for_last_10,away_overall_sot_against_last_10,away_overall_sot_diff_last_10,away_overall_corners_for_last_10,away_overall_corners_against_last_10,away_overall_corner_diff_last_10,away_overall_fouls_for_last_10,away_overall_fouls_against_last_10,away_overall_yellow_for_last_10,away_overall_yellow_against_last_10,away_overall_red_for_last_10,away_overall_red_against_last_10,away_venue_points_last_5,away_venue_win_last_5,away_venue_goals_for_last_5,away_venue_goals_against_last_5,away_venue_goal_diff_last_5,away_venue_shots_for_last_5,away_venue_shots_against_last_5,away_venue_sot_for_last_5,away_venue_sot_against_last_5,away_team_matches_played_before,away_days_since_last_match,diff_overall_points_last_5,diff_overall_points_last_10,diff_overall_win_last_5,diff_overall_goal_diff_last_5,diff_overall_goal_diff_last_10,diff_overall_goals_for_last_5,diff_overall_goals_against_last_5,diff_overall_shot_diff_last_5,diff_overall_sot_diff_last_5,diff_overall_corner_diff_last_5,diff_venue_points_last_5,diff_venue_goal_diff_last_5,diff_venue_sot_for_last_5,diff_venue_sot_against_last_5,diff_team_matches_played_before,diff_days_since_last_match,match_month,season_progress,referee_matches_before,referee_tota

## 15. Save Feature Dataset

This writes the engineered feature dataset to `data/processed/epl_features.csv` for modeling notebooks or scripts.

In [14]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
features_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved feature dataset to: {OUTPUT_PATH}")

Saved feature dataset to: ../data/processed/epl_features.csv


## 16. Summary

This notebook creates a first-pass pre-match feature dataset using:

- Overall rolling team form over the previous 5 and 10 matches.
- Venue-specific home and away form over the previous 5 matching-venue games.
- Rolling attacking and defensive metrics based on goals, shots, shots on target, and corners.
- Rolling discipline metrics based on fouls and cards.
- Rest-day features based on each team's previous match.
- Matchup differential features comparing the home team to the away team.
- Season timing features.
- Referee tendency features based on prior matches officiated.

The raw files also contain betting odds, which are valid pre-match information when available. They are not included in this first feature file because this version focuses on team-performance features, but odds can be useful later as a market baseline or benchmark.

The next step is to use `epl_features.csv` to train baseline classification models with a chronological train/test split.